In [ ]:
# ===============================
# PERPLEXITY-BASED DETECTORS
# ===============================

import pandas as pd
import numpy as np
import torch
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
)
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, brier_score_loss, roc_curve, accuracy_score
from sklearn.calibration import calibration_curve
import time
import warnings
warnings.filterwarnings('ignore')

# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ===============================
# STEP 1: LOAD PREPARED DATA
# ===============================
print("="*70)
print("STAGE 2C: PERPLEXITY-BASED DETECTORS")
print("="*70)

hc3_train  = pd.read_csv("hc3_train.csv")
hc3_test   = pd.read_csv("hc3_test.csv")
eli5_train = pd.read_csv("eli5_train.csv")
eli5_test  = pd.read_csv("eli5_test.csv")

print(f"✅ Data loaded:")
print(f"   HC3:  Train={len(hc3_train):,} | Test={len(hc3_test):,}")
print(f"   ELI5: Train={len(eli5_train):,} | Test={len(eli5_test):,}")

def encode_labels(labels):
    return (labels == 'llm').astype(int)

hc3_test['label_encoded']  = encode_labels(hc3_test['label'])
eli5_test['label_encoded'] = encode_labels(eli5_test['label'])

# ===============================
# CONFIG
# ===============================
MAX_LENGTH  = 512        # token window size
STRIDE      = 256        # sliding window stride
BATCH_SIZE  = 8          # texts per batch
PPL_CLIP    = 1e4        # outlier perplexity cap
NORM_METHOD = 'log_rank' # default normalization; swept across all methods below

REFERENCE_MODELS = {
    'GPT2-Small'   : 'gpt2',
    'GPT2-Medium'  : 'gpt2-medium',
    'GPT2-XL'      : 'gpt2-xl',
    'GPT-Neo-125M' : 'EleutherAI/gpt-neo-125m',
    'GPT-Neo-1.3B' : 'EleutherAI/gpt-neo-1.3b',
}

NORM_METHODS = ['rank', 'log_rank', 'minmax', 'sigmoid']

DATASETS = {
    'hc3' : hc3_test,
    'eli5': eli5_test,
}

print(f"\nConfig:")
print(f"  MAX_LENGTH={MAX_LENGTH} | STRIDE={STRIDE} | BATCH_SIZE={BATCH_SIZE}")
print(f"  PPL_CLIP={PPL_CLIP} | NORM_METHODS={NORM_METHODS}")
print(f"  Reference models: {list(REFERENCE_MODELS.keys())}")

# ===============================
# STEP 2: PERPLEXITY CALCULATOR
# ===============================
print("\n" + "="*70)
print("STEP 2: PERPLEXITY CALCULATION ENGINE")
print("="*70)

class PerplexityCalculator:
    """
    Calculates perplexity using a reference language model.
    Uses sliding window for texts longer than MAX_LENGTH to avoid
    silent truncation bias (LLM text tends to be longer).
    Clips extreme outlier perplexities to PPL_CLIP for rank stability.
    Lower perplexity = model finds text more predictable.
    """

    def __init__(self, model_name, device='cuda',
                 max_length=MAX_LENGTH, stride=STRIDE, ppl_clip=PPL_CLIP):
        print(f"Loading {model_name}...")
        self.model_name  = model_name
        self.device      = device
        self.max_length  = max_length
        self.stride      = stride
        self.ppl_clip    = ppl_clip

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model     = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
        ).to(device)
        self.model.eval()

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        print(f"✅ Loaded {model_name}")

    def calculate_perplexity(self, text):
        """
        Sliding window perplexity — handles texts longer than max_length.
        Without this, long LLM texts get silently truncated, biasing results.
        """
        encodings  = self.tokenizer(text, return_tensors='pt', truncation=False)
        input_ids  = encodings.input_ids.to(self.device)
        seq_len    = input_ids.shape[1]

        if seq_len == 0:
            return float('nan')

        # Short text — single forward pass
        if seq_len <= self.max_length:
            with torch.no_grad():
                outputs = self.model(input_ids, labels=input_ids)
            ppl = torch.exp(outputs.loss).item()
            return min(ppl, self.ppl_clip)

        # Long text — sliding window
        nlls       = []
        total_toks = 0
        for begin in range(0, seq_len, self.stride):
            end   = min(begin + self.max_length, seq_len)
            chunk = input_ids[:, begin:end]
            toks  = end - begin
            with torch.no_grad():
                outputs = self.model(chunk, labels=chunk)
            nlls.append(outputs.loss.item() * toks)
            total_toks += toks
            if end == seq_len:
                break

        avg_nll = sum(nlls) / total_toks
        ppl     = min(np.exp(avg_nll), self.ppl_clip)
        return ppl

    def calculate_batch_perplexity(self, texts, batch_size=BATCH_SIZE):
        """Calculate perplexity for a list of texts."""
        perplexities = []
        for i in tqdm(range(0, len(texts), batch_size),
                      desc=f"PPL ({self.model_name.split('/')[-1]})"):
            batch = texts[i:i + batch_size]
            for text in batch:
                try:
                    ppl = self.calculate_perplexity(str(text))
                    perplexities.append(ppl)
                except Exception as e:
                    print(f"  ⚠ Error on sample {i}: {e}")
                    perplexities.append(np.nan)
        return np.array(perplexities)

    def cleanup(self):
        """Free GPU memory before loading next model."""
        del self.model
        torch.cuda.empty_cache()
        print(f"  🧹 {self.model_name} unloaded from GPU")

# ===============================
# STEP 3: REFERENCE MODEL INFO
# ===============================
print("\n" + "="*70)
print("STEP 3: REFERENCE LANGUAGE MODELS")
print("="*70)

model_vram = {
    'GPT2-Small'   : '~0.25 GB',
    'GPT2-Medium'  : '~0.6 GB',
    'GPT2-XL'      : '~3.0 GB',
    'GPT-Neo-125M' : '~0.3 GB',
    'GPT-Neo-1.3B' : '~3.0 GB',
}
for name, ckpt in REFERENCE_MODELS.items():
    print(f"  {name:15s} | {ckpt:35s} | VRAM(fp16): {model_vram[name]}")

# ===============================
# STEP 4: CALCULATE PERPLEXITIES
# ===============================
print("\n" + "="*70)
print("STEP 4: CALCULATING PERPLEXITIES")
print("="*70)

perplexity_results = {}

for model_name, model_checkpoint in REFERENCE_MODELS.items():
    print(f"\n{'='*70}")
    print(f"MODEL: {model_name}  ({model_checkpoint})")
    print(f"{'='*70}")
    start = time.time()

    calculator = PerplexityCalculator(model_checkpoint, device=device)
    perplexity_results[model_name] = {}

    for ds_name, ds_df in DATASETS.items():
        print(f"\n[{ds_name.upper()} Test Set — {len(ds_df)} samples]")
        ppls = calculator.calculate_batch_perplexity(
            ds_df['text'].tolist(), batch_size=BATCH_SIZE)

        perplexity_results[model_name][ds_name] = {
            'perplexities': ppls,
            'labels':       ds_df['label_encoded'].values
        }

        valid = ~np.isnan(ppls)
        human_ppl = ppls[valid & (ds_df['label_encoded'].values == 0)]
        llm_ppl   = ppls[valid & (ds_df['label_encoded'].values == 1)]
        print(f"  Human median PPL : {np.median(human_ppl):.2f}")
        print(f"  LLM   median PPL : {np.median(llm_ppl):.2f}")
        print(f"  NaN count        : {(~valid).sum()}")

    calculator.cleanup()
    elapsed = time.time() - start
    print(f"  ⏱ {model_name} total: {elapsed/60:.1f} min")

# ===============================
# STEP 5: CONVERT PERPLEXITY TO DETECTABILITY SCORES
# ===============================
print("\n" + "="*70)
print("STEP 5: CONVERTING PERPLEXITY TO DETECTABILITY SCORES")
print("="*70)

def perplexity_to_detectability(perplexities, labels, method='log_rank'):
    """
    Convert raw perplexity to [0,1] P(llm) detectability score.

    KEY INSIGHT (bug fix from v1):
      GPT2/GPT-Neo assign LOWER perplexity to LLM text (both are transformer LMs).
      So higher PPL → more human-like → LOWER P(llm).
      All methods invert the relationship: high PPL → low detectability score.

    Methods:
      rank     : rank-based, robust to outliers
      log_rank : rank on log(PPL), more robust to extreme spikes
      minmax   : linear rescaling, sensitive to outliers
      sigmoid  : sigmoid around median, smooth calibration
    """
    valid_mask    = ~np.isnan(perplexities)
    ppl_clean     = perplexities[valid_mask]
    labels_clean  = labels[valid_mask]

    if method == 'rank':
        ranks = np.argsort(np.argsort(ppl_clean))
        # INVERTED: high PPL rank → low LLM score
        detectability = 1.0 - (ranks / (len(ranks) - 1))

    elif method == 'log_rank':
        log_ppl = np.log(ppl_clean + 1e-8)
        ranks   = np.argsort(np.argsort(log_ppl))
        detectability = 1.0 - (ranks / (len(ranks) - 1))

    elif method == 'minmax':
        ppl_min = ppl_clean.min()
        ppl_max = ppl_clean.max()
        # INVERTED: high PPL → low score
        detectability = 1.0 - (ppl_clean - ppl_min) / (ppl_max - ppl_min + 1e-8)

    elif method == 'sigmoid':
        median = np.median(ppl_clean)
        scale  = np.std(ppl_clean) + 1e-8
        # INVERTED: positive deviation from median → human → low LLM score
        detectability = 1.0 / (1.0 + np.exp((ppl_clean - median) / scale))

    else:
        raise ValueError(f"Unknown method: {method}")

    detectability = np.clip(detectability, 0.0, 1.0)
    return detectability, labels_clean, valid_mask


def best_threshold_accuracy(y_true, scores):
    """Find optimal decision threshold via Youden's J statistic."""
    fpr, tpr, thresholds = roc_curve(y_true, scores)
    j_scores  = tpr - fpr
    best_idx  = j_scores.argmax()
    best_thresh = float(thresholds[best_idx])
    preds = (scores >= best_thresh).astype(int)
    acc   = accuracy_score(y_true, preds)
    return best_thresh, acc


# Compute detectability for all methods, store best per model/dataset
detectability_results = {}
method_comparison_rows = []

for model_name, datasets in perplexity_results.items():
    detectability_results[model_name] = {}

    for ds_name, data in datasets.items():
        best_auc    = -1
        best_method = None
        best_result = None

        print(f"\n{model_name} | {ds_name.upper()}")

        for method in NORM_METHODS:
            det, labels_clean, valid_mask = perplexity_to_detectability(
                data['perplexities'], data['labels'], method=method)

            if len(np.unique(labels_clean)) < 2:
                continue

            auc    = roc_auc_score(labels_clean, det)
            brier  = brier_score_loss(labels_clean, det)
            thresh, acc = best_threshold_accuracy(labels_clean, det)

            print(f"  {method:10s}: AUC={auc:.4f}  Brier={brier:.4f}  "
                  f"Acc@opt={acc:.4f}  thresh={thresh:.3f}")

            method_comparison_rows.append({
                'Model':   model_name,
                'Dataset': ds_name.upper(),
                'Method':  method,
                'ROC-AUC': round(auc, 4),
                'Brier':   round(brier, 4),
                'Acc@opt': round(acc, 4),
                'Thresh':  round(thresh, 4),
            })

            if auc > best_auc:
                best_auc    = auc
                best_method = method
                best_result = {
                    'detectability_scores': det,
                    'y_true':               labels_clean,
                    'y_pred':               (det >= thresh).astype(int),
                    'raw_perplexities':     data['perplexities'][valid_mask],
                    'roc_auc':              auc,
                    'brier_score':          brier,
                    'best_thresh':          thresh,
                    'best_acc':             acc,
                    'best_method':          best_method,
                }

        print(f"  → Best method: {best_method} (AUC={best_auc:.4f})")
        detectability_results[model_name][ds_name] = best_result

        # Save scores using npz (bug fix — np.save can't handle dicts)
        safe_model = model_name.replace(' ', '_').replace('-', '_')
        np.savez(
            f"ppl_scores_{safe_model}_{ds_name}.npz",
            y_true=labels_clean,
            y_score=det
        )

print("\n✅ Perplexity → Detectability conversion complete")

# ===============================
# STEP 6: CALCULATE METRICS
# ===============================
print("\n" + "="*70)
print("STEP 6: PERFORMANCE METRICS")
print("="*70)

for model_name, datasets in detectability_results.items():
    print(f"\n{model_name}:")
    for ds_name, data in datasets.items():
        print(f"  {ds_name}: ROC-AUC={data['roc_auc']:.4f}  "
              f"Brier={data['brier_score']:.4f}  "
              f"Acc@opt={data['best_acc']:.4f}  "
              f"thresh={data['best_thresh']:.3f}  "
              f"method={data['best_method']}")

# ===============================
# STEP 7: PERPLEXITY DISTRIBUTIONS
# ===============================
print("\n" + "="*70)
print("STEP 7: PERPLEXITY DISTRIBUTION ANALYSIS")
print("="*70)

n_models = len(REFERENCE_MODELS)
fig, axes = plt.subplots(n_models, len(DATASETS),
                         figsize=(14, 4 * n_models))

for i, (model_name, datasets) in enumerate(detectability_results.items()):
    for j, (ds_name, data) in enumerate(datasets.items()):
        ax = axes[i, j] if n_models > 1 else axes[j]

        human_ppl = data['raw_perplexities'][data['y_true'] == 0]
        llm_ppl   = data['raw_perplexities'][data['y_true'] == 1]

        ax.hist(np.log10(human_ppl + 1e-8), bins=50, alpha=0.6,
                label='Human', color='blue')
        ax.hist(np.log10(llm_ppl + 1e-8),   bins=50, alpha=0.6,
                label='LLM',   color='red')

        ax.axvline(np.log10(np.median(human_ppl) + 1e-8),
                   color='blue', linestyle='--', alpha=0.7,
                   label=f'Human median={np.median(human_ppl):.1f}')
        ax.axvline(np.log10(np.median(llm_ppl) + 1e-8),
                   color='red',  linestyle='--', alpha=0.7,
                   label=f'LLM median={np.median(llm_ppl):.1f}')

        ax.set_xlabel('Log10(Perplexity)')
        ax.set_ylabel('Frequency')
        ax.set_title(f'{model_name} - {ds_name.upper()}\n'
                     f'Perplexity Distribution')
        ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig('ppl_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

# ===============================
# STEP 8: DETECTABILITY SCORE DISTRIBUTIONS
# ===============================
print("\n" + "="*70)
print("STEP 8: DETECTABILITY SCORE DISTRIBUTIONS")
print("="*70)

fig, axes = plt.subplots(n_models, len(DATASETS),
                         figsize=(14, 4 * n_models))

for i, (model_name, datasets) in enumerate(detectability_results.items()):
    for j, (ds_name, data) in enumerate(datasets.items()):
        ax = axes[i, j] if n_models > 1 else axes[j]

        human_scores = data['detectability_scores'][data['y_true'] == 0]
        llm_scores   = data['detectability_scores'][data['y_true'] == 1]

        ax.hist(human_scores, bins=30, alpha=0.6, label='Human',
                color='blue', range=(0, 1))
        ax.hist(llm_scores,   bins=30, alpha=0.6, label='LLM',
                color='red',  range=(0, 1))

        ax.axvline(0.5, color='black', linestyle='--',
                   alpha=0.4, label='thresh=0.5')
        ax.axvline(data['best_thresh'], color='green', linestyle='-',
                   alpha=0.7, label=f"opt={data['best_thresh']:.2f}")

        ax.set_xlabel('Detectability Score  [P(llm)]')
        ax.set_ylabel('Frequency')
        ax.set_title(f'{model_name} - {ds_name.upper()}\n'
                     f'Score Distribution  (method={data["best_method"]}  '
                     f'AUC={data["roc_auc"]:.3f})')
        ax.legend(fontsize=7)
        ax.set_xlim(0, 1)

plt.tight_layout()
plt.savefig('ppl_score_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

# ===============================
# STEP 9: CALIBRATION CURVES
# ===============================
print("\n" + "="*70)
print("STEP 9: CALIBRATION ANALYSIS")
print("="*70)

fig, axes = plt.subplots(n_models, len(DATASETS),
                         figsize=(14, 4 * n_models))

for i, (model_name, datasets) in enumerate(detectability_results.items()):
    for j, (ds_name, data) in enumerate(datasets.items()):
        ax = axes[i, j] if n_models > 1 else axes[j]

        fraction_pos, mean_pred = calibration_curve(
            data['y_true'],
            data['detectability_scores'],
            n_bins=10,
            strategy='uniform'
        )

        ax.plot(mean_pred, fraction_pos, 's-',
                label='Detector', linewidth=2)
        ax.plot([0, 1], [0, 1], 'k--',
                label='Perfect', alpha=0.5)
        ax.set_xlabel('Mean Predicted Detectability')
        ax.set_ylabel('Fraction of LLM Text')
        ax.set_title(f'{model_name} - {ds_name.upper()}\nCalibration')
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('ppl_calibration.png', dpi=150, bbox_inches='tight')
plt.show()

# ===============================
# STEP 10: CROSS-MODEL COMPARISON
# ===============================
print("\n" + "="*70)
print("STEP 10: CROSS-MODEL PERPLEXITY COMPARISON")
print("="*70)

comparison_data = []

for model_name, datasets in detectability_results.items():
    for ds_name, data in datasets.items():
        human_ppl = data['raw_perplexities'][data['y_true'] == 0]
        llm_ppl   = data['raw_perplexities'][data['y_true'] == 1]

        comparison_data.append({
            'Model':             model_name,
            'Dataset':           ds_name.upper(),
            'Human_Median_PPL':  np.median(human_ppl),
            'LLM_Median_PPL':    np.median(llm_ppl),
            'PPL_Ratio':         np.median(llm_ppl) / (np.median(human_ppl) + 1e-8),
            'ROC-AUC':           data['roc_auc'],
            'Brier_Score':       data['brier_score'],
            'Best_Method':       data['best_method'],
        })

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.round(4).to_string(index=False))

# ===============================
# STEP 11: OVERLAP ANALYSIS
# ===============================
print("\n" + "="*70)
print("STEP 11: DETECTABILITY OVERLAP ANALYSIS")
print("="*70)

overlap_data = []

for model_name, datasets in detectability_results.items():
    for ds_name, data in datasets.items():
        human_scores = data['detectability_scores'][data['y_true'] == 0]
        llm_scores   = data['detectability_scores'][data['y_true'] == 1]

        human_unc = ((human_scores >= 0.4) & (human_scores <= 0.6)).sum()
        llm_unc   = ((llm_scores   >= 0.4) & (llm_scores   <= 0.6)).sum()

        overlap_data.append({
            'Model':               model_name,
            'Dataset':             ds_name.upper(),
            'Mean_Human_Score':    human_scores.mean(),
            'Mean_LLM_Score':      llm_scores.mean(),
            'Score_Separation':    llm_scores.mean() - human_scores.mean(),
            'Human_Uncertain_%':   (human_unc / len(human_scores)) * 100,
            'LLM_Uncertain_%':     (llm_unc   / len(llm_scores))   * 100,
        })

overlap_df = pd.DataFrame(overlap_data)
print(overlap_df.round(4).to_string(index=False))

# ===============================
# STEP 12: SUMMARY TABLE
# ===============================
print("\n" + "="*70)
print("STEP 12: PERFORMANCE SUMMARY")
print("="*70)

summary_rows = []

for model_name, datasets in detectability_results.items():
    for ds_name, data in datasets.items():
        human_scores = data['detectability_scores'][data['y_true'] == 0]
        llm_scores   = data['detectability_scores'][data['y_true'] == 1]

        summary_rows.append({
            'Detector':          f"PPL-{model_name}",
            'Evaluation':        ds_name,
            'Method':            data['best_method'],
            'ROC-AUC':           data['roc_auc'],
            'Brier Score':       data['brier_score'],
            'Acc@Optimal':       data['best_acc'],
            'Opt_Threshold':     data['best_thresh'],
            'Mean Human Score':  human_scores.mean(),
            'Mean LLM Score':    llm_scores.mean(),
            'Score Separation':  llm_scores.mean() - human_scores.mean(),
        })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.round(4).to_string(index=False))

# Method sweep comparison table
print("\n── Normalization Method Comparison ──")
method_df = pd.DataFrame(method_comparison_rows)
print(method_df.round(4).to_string(index=False))

# ===============================
# SAVE ALL RESULTS
# ===============================
summary_df.to_csv("detector_family3_perplexity_results.csv",    index=False)
comparison_df.to_csv("detector_family3_perplexity_comparison.csv", index=False)
overlap_df.to_csv("detector_family3_overlap_analysis.csv",         index=False)
method_df.to_csv("detector_family3_method_comparison.csv",         index=False)

# ===============================
# STEP 13: KEY INSIGHTS
# ===============================
print("\n" + "="*70)
print("✅ COMPLETE - PERPLEXITY-BASED DETECTORS")
print("="*70)


print("NORMALIZATION METHOD SWEEP:")
for method in NORM_METHODS:
    sub = method_df[method_df['Method'] == method]
    if len(sub):
        mean_auc = sub['ROC-AUC'].mean()
        print(f"   {method:10s}: mean AUC across all conditions = {mean_auc:.4f}")

print("CROSS-DATASET PERFORMANCE:")
for model_name in detectability_results.keys():
    hc3_auc  = detectability_results[model_name]['hc3']['roc_auc']
    eli5_auc = detectability_results[model_name]['eli5']['roc_auc']
    print(f"   {model_name}: HC3={hc3_auc:.3f}, ELI5={eli5_auc:.3f}")